# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [58]:
import sys
print(sys.executable)

/Users/sai/Desktop/tinyml-arduino/bin/python


In [59]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [60]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [61]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [62]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [63]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#


X = df.drop(columns=["Class"]).values
y = df["Class"].values

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (178, 13)
y shape: (178,)


In [64]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
)

print("Training data shape:", X_train.shape)
print("Training labels shape:", y_train.shape)
print("Test data shape:", X_test.shape)
print("Test labels shape:", y_test.shape)


Training data shape: (124, 13)
Training labels shape: (124,)
Test data shape: (54, 13)
Test labels shape: (54,)


In [65]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled X_train → mean per feature:", X_train_scaled.mean(axis=0).round(4))
print("Scaled X_train → std per feature: ", X_train_scaled.std(axis=0).round(4))

Scaled X_train → mean per feature: [ 0. -0.  0. -0.  0.  0.  0. -0.  0.  0.  0.  0.  0.]
Scaled X_train → std per feature:  [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [66]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat  = to_categorical(y_test,  num_classes=num_classes)

print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat shape: ", y_test_cat.shape)


y_train_cat shape: (124, 3)
y_test_cat shape:  (54, 3)


In [67]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.summary()

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_21 (Dense)            (None, 64)                896       
                                                                 
 dense_22 (Dense)            (None, 32)                2080      
                                                                 
 dense_23 (Dense)            (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [68]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#


model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_scaled, y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)


Epoch 1/20
13/13 [==============================] - 0s 3ms/step - loss: 0.9447 - accuracy: 0.5657 - val_loss: 0.7157 - val_accuracy: 0.8000
Epoch 2/20
13/13 [==============================] - 0s 840us/step - loss: 0.6489 - accuracy: 0.9192 - val_loss: 0.5264 - val_accuracy: 0.9200
Epoch 3/20
13/13 [==============================] - 0s 824us/step - loss: 0.4670 - accuracy: 0.9697 - val_loss: 0.3939 - val_accuracy: 1.0000
Epoch 4/20
13/13 [==============================] - 0s 754us/step - loss: 0.3435 - accuracy: 0.9697 - val_loss: 0.3060 - val_accuracy: 1.0000
Epoch 5/20
13/13 [==============================] - 0s 752us/step - loss: 0.2560 - accuracy: 0.9697 - val_loss: 0.2429 - val_accuracy: 1.0000
Epoch 6/20
13/13 [==============================] - 0s 735us/step - loss: 0.1918 - accuracy: 0.9697 - val_loss: 0.1912 - val_accuracy: 1.0000
Epoch 7/20
13/13 [==============================] - 0s 736us/step - loss: 0.1483 - accuracy: 0.9798 - val_loss: 0.1560 - val_accuracy: 1.0000
Epoch 8/

In [69]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#

test_loss, test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
y_pred = np.argmax(model.predict(X_test_scaled), axis=1)
y_true = np.argmax(y_test_cat, axis=1)


print("test loss:", test_loss)
print("test acc:", test_acc) 

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))


2/2 [==============================] - 0s 825us/step
test loss: 0.029928982257843018
test acc: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


In [70]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#

import os

tflite_path = "model_base.tflite"

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

size_kb = os.path.getsize(tflite_path) / 1024
print(f"Base TFLite model size: {size_kb:.2f} KB")






INFO:tensorflow:Assets written to: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmp1c5pc3go/assets


INFO:tensorflow:Assets written to: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmp1c5pc3go/assets


Base TFLite model size: 14.14 KB


2026-05-20 13:19:07.822060: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 13:19:07.822070: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 13:19:07.822139: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmp1c5pc3go
2026-05-20 13:19:07.822470: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 13:19:07.822473: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmp1c5pc3go
2026-05-20 13:19:07.823295: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 13:19:07.835486: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmp1c5pc3go
2026-05-

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [71]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]      

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    # <-- Enter your code here <--
    
    tflite_model = converter.convert()
    
    with open(filename, 'wb') as f:
        
        f.write(tflite_model)


    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    # <-- Enter your code here for TFLite inference <--#
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    in_det = input_details[0]
    out_det = output_details[0]

    y_pred = []
    for sample in X_test:
        data = sample[np.newaxis, :].astype(np.float32)

        if in_det['dtype'] == np.int8:
            sc, zp = in_det['quantization']
            data = (data / sc + zp).astype(np.int8)

        interpreter.set_tensor(in_det['index'], data)
        interpreter.invoke()
        result = interpreter.get_tensor(out_det['index'])

        if out_det['dtype'] == np.int8:
            sc, zp = out_det['quantization']
            result = (result.astype(np.float32) - zp) * sc

        y_pred.append(np.argmax(result))

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)
    

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")
    print(f"{quant_type.upper()} Accuracy: {np.mean(y_pred == y_true):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    # <-- Enter your code here: print classification_report and confusion_matrix <--#


In [72]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#
def file_size_kb(path):
    return os.path.getsize(path) / 1024.0

    
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'dynamic', 'model_dynamic.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'int8', 'model_int8.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'float16', 'model_float16.tflite')


INFO:tensorflow:Assets written to: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpb3rb02aq/assets


INFO:tensorflow:Assets written to: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpb3rb02aq/assets



DYNAMIC TFLite model size: 8.24 KB
DYNAMIC Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]
INFO:tensorflow:Assets written to: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpblbs9xr1/assets


2026-05-20 13:19:09.917372: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 13:19:09.917381: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 13:19:09.917442: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpb3rb02aq
2026-05-20 13:19:09.917778: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 13:19:09.917781: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpb3rb02aq
2026-05-20 13:19:09.918637: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 13:19:09.931068: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpb3rb02aq
2026-05-


INT8 TFLite model size: 5.82 KB
INT8 Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]
INFO:tensorflow:Assets written to: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpzsrfmyvt/assets


INFO:tensorflow:Assets written to: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpzsrfmyvt/assets
2026-05-20 13:19:10.413011: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 13:19:10.413021: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.



FLOAT16 TFLite model size: 9.04 KB
FLOAT16 Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


2026-05-20 13:19:10.413087: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpzsrfmyvt
2026-05-20 13:19:10.413349: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 13:19:10.413352: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpzsrfmyvt
2026-05-20 13:19:10.414177: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 13:19:10.426291: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpzsrfmyvt
2026-05-20 13:19:10.429746: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 16657 microseconds.


## Problem 1 - Part (c)

### Pruning

In [73]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#
batch_size = 8
epochs_prune = 10
end_step = (X_train_scaled.shape[0] // batch_size) * epochs_prune

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)
print(f"end_step: {end_step}")





end_step: 150


In [74]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#
prune = tfmot.sparsity.keras.prune_low_magnitude
input_dim = X_train_scaled.shape[1]
model_pruned = Sequential([
    prune(Dense(64, activation='relu', input_shape=(input_dim,)), pruning_schedule=pruning_schedule),
    prune(Dense(32, activation='relu'), pruning_schedule=pruning_schedule),
    prune(Dense(num_classes, activation='softmax'), pruning_schedule=pruning_schedule),
])

model_pruned.summary()


Model: "sequential_8"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense_  (None, 64)                1730      
 24 (PruneLowMagnitude)                                          
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 25 (PruneLowMagnitude)                                          
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 26 (PruneLowMagnitude)                                          
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [75]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#

model_pruned.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

p_callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

p_history = model_pruned.fit(
    X_train_scaled, y_train_cat, epochs=10, batch_size=8, validation_split=0.2, callbacks=p_callbacks,verbose=1
)



Epoch 1/10
13/13 [==============================] - 0s 4ms/step - loss: 1.0155 - accuracy: 0.5152 - val_loss: 0.8472 - val_accuracy: 0.8000
Epoch 2/10
13/13 [==============================] - 0s 927us/step - loss: 0.7660 - accuracy: 0.8586 - val_loss: 0.6440 - val_accuracy: 0.9600
Epoch 3/10
13/13 [==============================] - 0s 938us/step - loss: 0.5924 - accuracy: 0.9293 - val_loss: 0.4878 - val_accuracy: 0.9600
Epoch 4/10
13/13 [==============================] - 0s 882us/step - loss: 0.4447 - accuracy: 0.9394 - val_loss: 0.3633 - val_accuracy: 0.9600
Epoch 5/10
13/13 [==============================] - 0s 830us/step - loss: 0.3297 - accuracy: 0.9596 - val_loss: 0.2737 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 827us/step - loss: 0.2401 - accuracy: 0.9798 - val_loss: 0.2109 - val_accuracy: 1.0000
Epoch 7/10
13/13 [==============================] - 0s 800us/step - loss: 0.1816 - accuracy: 0.9697 - val_loss: 0.1703 - val_accuracy: 1.0000
Epoch 8/

In [76]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#

stripped = tfmot.sparsity.keras.strip_pruning(model_pruned)

tflite_converter = tf.lite.TFLiteConverter.from_keras_model(stripped)
tflite_converter.optimizations = [tf.lite.Optimize.DEFAULT]
pruned_tflite = tflite_converter.convert()

pruned_path = 'model_pruned.tflite'
with open(pruned_path, 'wb') as f:
    f.write(pruned_tflite)

print(f"Pruned TFLite model size: {file_size_kb(pruned_path):.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmp438lawpn/assets


INFO:tensorflow:Assets written to: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmp438lawpn/assets


Pruned TFLite model size: 8.28 KB


2026-05-20 13:19:14.557980: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 13:19:14.557990: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 13:19:14.558053: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmp438lawpn
2026-05-20 13:19:14.558239: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 13:19:14.558241: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmp438lawpn
2026-05-20 13:19:14.558726: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 13:19:14.563570: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmp438lawpn
2026-05-

In [77]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#

y_true = np.argmax(y_test_cat, axis=1)
y_pred_pruned = np.argmax(stripped.predict(X_test_scaled), axis=1)

print("\nClassification Report (Pruned):")
print(classification_report(y_true, y_pred_pruned))
print("Confusion Matrix (Pruned):")
print(confusion_matrix(y_true, y_pred_pruned))

2/2 [==============================] - 0s 2ms/step

Classification Report (Pruned):
              precision    recall  f1-score   support

           0       1.00      0.95      0.97        19
           1       0.95      1.00      0.98        21
           2       1.00      1.00      1.00        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix (Pruned):
[[18  1  0]
 [ 0 21  0]
 [ 0  0 14]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [78]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#

student_model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

student_model.summary()

Model: "sequential_9"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_27 (Dense)            (None, 32)                448       
                                                                 
 dense_28 (Dense)            (None, 16)                528       
                                                                 
 dense_29 (Dense)            (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [79]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
soft_labels = model.predict(X_train_scaled)

print("Soft label shape:", soft_labels.shape)
print("Sample soft label:", soft_labels[0])

4/4 [==============================] - 0s 791us/step
Soft label shape: (124, 3)
Sample soft label: [0.00099101 0.02608333 0.97292566]


In [80]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# <-- Enter your code here <--#

y_train_combined = np.concatenate([y_train_cat, soft_labels], axis=1)
print("Combined label shape:", y_train_combined.shape)

alpha = 0.5

def distillation_loss(y_true_combined, y_pred):
    y_hard = y_true_combined[:, :num_classes]
    y_soft = y_true_combined[:, num_classes:]

    loss_hard = tf.keras.losses.categorical_crossentropy(y_hard, y_pred)
    loss_soft = tf.keras.losses.categorical_crossentropy(y_soft, y_pred)

    return alpha * loss_hard + (1 - alpha) * loss_soft

    

Combined label shape: (124, 6)


In [81]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#

student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

history_kd = student_model.fit(
    X_train_scaled, y_train_combined, epochs=10, batch_size=8, validation_split=0.2,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 0s 4ms/step - loss: 1.1735 - accuracy: 0.4343 - val_loss: 1.0651 - val_accuracy: 0.3600
Epoch 2/10
13/13 [==============================] - 0s 849us/step - loss: 0.9964 - accuracy: 0.5455 - val_loss: 0.9176 - val_accuracy: 0.6400
Epoch 3/10
13/13 [==============================] - 0s 849us/step - loss: 0.8553 - accuracy: 0.7071 - val_loss: 0.7851 - val_accuracy: 0.8400
Epoch 4/10
13/13 [==============================] - 0s 750us/step - loss: 0.7281 - accuracy: 0.8687 - val_loss: 0.6626 - val_accuracy: 0.8400
Epoch 5/10
13/13 [==============================] - 0s 761us/step - loss: 0.6068 - accuracy: 0.9091 - val_loss: 0.5519 - val_accuracy: 0.8800
Epoch 6/10
13/13 [==============================] - 0s 769us/step - loss: 0.4956 - accuracy: 0.9495 - val_loss: 0.4538 - val_accuracy: 0.9200
Epoch 7/10
13/13 [==============================] - 0s 719us/step - loss: 0.3984 - accuracy: 0.9495 - val_loss: 0.3716 - val_accuracy: 0.9600
Epoch 8/

In [82]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# <-- Enter your code here <--#
kd_path = 'model_kd.tflite'

kd_converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
kd_model = kd_converter.convert()

with open(kd_path, 'wb') as f:
    f.write(kd_model)

print(f"KD TFLite model size: {file_size_kb(kd_path):.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpvm6q5pdg/assets


INFO:tensorflow:Assets written to: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpvm6q5pdg/assets


KD TFLite model size: 6.14 KB


2026-05-20 13:19:19.499881: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 13:19:19.499891: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 13:19:19.499954: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpvm6q5pdg
2026-05-20 13:19:19.500230: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 13:19:19.500233: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpvm6q5pdg
2026-05-20 13:19:19.501051: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 13:19:19.513877: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpvm6q5pdg
2026-05-

In [83]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#

y_true = np.argmax(y_test_cat, axis=1)
y_pred_kd = np.argmax(student_model.predict(X_test_scaled), axis=1)

print("\nClassification Report (KD Student):")
print(classification_report(y_true, y_pred_kd))
print("Confusion Matrix (KD Student):")
print(confusion_matrix(y_true, y_pred_kd))


2/2 [==============================] - 0s 1ms/step

Classification Report (KD Student):
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        19
           1       1.00      0.90      0.95        21
           2       0.93      1.00      0.97        14

    accuracy                           0.96        54
   macro avg       0.96      0.97      0.96        54
weighted avg       0.97      0.96      0.96        54

Confusion Matrix (KD Student):
[[19  0  0]
 [ 1 19  1]
 [ 0  0 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [85]:
# <-- (if needed) Enter your code here <--#
# Build a smaller student model than part (d)
# <-- (if needed) Enter your code here <--#
# Problem 1(e): Further compression using KD + INT8 quantization

kd_int8_filename = "model_kd_int8.tflite"

student_converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
student_converter.optimizations = [tf.lite.Optimize.DEFAULT]
student_converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
student_converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
student_converter.inference_input_type = tf.int8
student_converter.inference_output_type = tf.int8

student_quantized_model = student_converter.convert()

with open(kd_int8_filename, "wb") as f:
    f.write(student_quantized_model)

print("KD + INT8 model saved as:", kd_int8_filename)
print(f"KD + INT8 model size: {file_size_kb(kd_int8_filename):.2f} KB")


kd_int8_interpreter = tf.lite.Interpreter(model_path=kd_int8_filename)
kd_int8_interpreter.allocate_tensors()

input_details = kd_int8_interpreter.get_input_details()[0]
output_details = kd_int8_interpreter.get_output_details()[0]

input_scale, input_zero_point = input_details["quantization"]
output_scale, output_zero_point = output_details["quantization"]


kd_int8_predictions = []

for x in X_test_scaled:
    x = np.expand_dims(x, axis=0).astype(np.float32)

    if input_details["dtype"] == np.int8:
        x = (x / input_scale + input_zero_point).astype(np.int8)

    kd_int8_interpreter.set_tensor(input_details["index"], x)
    kd_int8_interpreter.invoke()

    pred = kd_int8_interpreter.get_tensor(output_details["index"])

    if output_details["dtype"] == np.int8:
        pred = (pred.astype(np.float32) - output_zero_point) * output_scale

    kd_int8_predictions.append(np.argmax(pred))

kd_int8_predictions = np.array(kd_int8_predictions)
actual_labels = np.argmax(y_test_cat, axis=1)


kd_int8_accuracy = np.mean(kd_int8_predictions == actual_labels)

print(f"\nKD + INT8 Accuracy: {kd_int8_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(actual_labels, kd_int8_predictions))
print("Confusion Matrix:")
print(confusion_matrix(actual_labels, kd_int8_predictions))


model_files = {
    "Base float32": "model_base.tflite",
    "Dynamic quantization": "model_dynamic.tflite",
    "Full INT8 quantization": "model_int8.tflite",
    "Float16 quantization": "model_float16.tflite",
    "Pruned model": "model_pruned.tflite",
    "KD student": "model_kd.tflite",
    "KD + INT8": kd_int8_filename
}

print("\nModel Size Comparison:")
for model_name, model_path in model_files.items():
    print(f"{model_name:28s}: {file_size_kb(model_path):.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpwuslp8jj/assets


INFO:tensorflow:Assets written to: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpwuslp8jj/assets
/Users/sai/Desktop/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 15:26:41.933824: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 15:26:41.933952: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.


KD + INT8 model saved as: model_kd_int8.tflite
KD + INT8 model size: 3.68 KB

KD + INT8 Accuracy: 0.9630

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        19
           1       1.00      0.90      0.95        21
           2       0.93      1.00      0.97        14

    accuracy                           0.96        54
   macro avg       0.96      0.97      0.96        54
weighted avg       0.97      0.96      0.96        54

Confusion Matrix:
[[19  0  0]
 [ 1 19  1]
 [ 0  0 14]]

Model Size Comparison:
Base float32                : 14.14 KB
Dynamic quantization        : 8.24 KB
Full INT8 quantization      : 5.82 KB
Float16 quantization        : 9.04 KB
Pruned model                : 8.28 KB
KD student                  : 6.14 KB
KD + INT8                   : 3.68 KB


2026-05-20 15:26:41.935435: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpwuslp8jj
2026-05-20 15:26:41.937606: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 15:26:41.937609: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpwuslp8jj
2026-05-20 15:26:41.944179: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 15:26:41.991699: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/dv/_kb6lsq96b7cd_1b9l34scxr0000gp/T/tmpwuslp8jj
2026-05-20 15:26:41.995622: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 60187 microseconds.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
